This notebook contains the process of collecting advice-seeking prompts across four different professional domains from four different source datasets accessed through HuggingFace. Since all datasets did not only include advice-seeking prompts, keyword filtering was performed. Filtering was followed by random sampling 300 examples from each domain.

In [ ]:
from datasets import load_dataset, concatenate_datasets, load_from_disk
import random
import re

In [ ]:
!pip install --quiet huggingface_hub

In [ ]:
from huggingface_hub import login

In [ ]:
login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
# keywords for finding advice-seeking prompts
ADVICE_KEYWORDS = [
    "should I", "what should I do", "any advice", "help me",
    "how do I", "how can I", "what's the best way",
    "recommend", "suggest", "tips", "guidance"
]
ADVICE_KEYWORDS = [keyword.lower() for keyword in ADVICE_KEYWORDS]

# function for loading and cleaning datasets
def load_and_clean(name, text_column):
    ds = load_dataset(name)

    # merge possible train/test splits
    ds = concatenate_datasets([ds[split] for split in ds.keys()])

    # keep only input column
    ds = ds.remove_columns(
        [col for col in ds.column_names if col != text_column]
    )
    return ds

# function for checking if example contains advice-seeking keywords
def is_advice_prompt(text):
  if text is None:
      return False
  text_lower = text.lower()
  return any(keyword in text_lower for keyword in ADVICE_KEYWORDS)

# function for filtering advice-seeking -dataset out of original dataset
def filter_advice_dataset(dataset, text_column):
    return dataset.filter(lambda x: is_advice_prompt(x[text_column]))

# 1. Medical prompts dataset

In [ ]:
dataset = load_and_clean("lavita/ChatDoctor-HealthCareMagic-100k", "input")

README.md:   0%|          | 0.00/542 [00:00<?, ?B/s]

data/train-00000-of-00001-5e7cb295b9cff0(…):   0%|          | 0.00/70.5M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/112165 [00:00<?, ? examples/s]

In [ ]:
dataset

Dataset({
    features: ['input'],
    num_rows: 112165
})

In [ ]:
ds_filtered = filter_advice_dataset(dataset, text_column="input")

In [ ]:
ds_filtered

Dataset({
    features: ['input'],
    num_rows: 27155
})

In [ ]:
# sample 300 examples
ds_sampled = ds_filtered.shuffle(42).select(range(300))
ds_sampled[:5]

{'input': ['Hi. Im having a fever which has lasted for 3 days as of now, it doesnt seems to subside with Paracetamol, temperature is about 37.5 degree Celsius. Besides, I feel very tired and sleepy even though I have just woke up from a nap. The doctor has took some blood for Full Blood Count test, but the result is not out yet. Is it necessarily to go to the hospital, or what should I do? Please help.',
  'I feel from a tree a couple of weeks ago onto my right hip. I still have some local bruising but have a large lump with local discomfort. Also it feals like simethinh is missing inside my outer hip area as compared to opposite hip. Is there anything to be cincerned about and should I seek medical attention?',
  'My 84 year old mother has recently ( last 4 days) been experiencing sharp pain in the left side of her abdomen , at about the same hight as her belly button. The pain is so severe that she yells out in pain. It goes away in a few seconds and she feels OK, however it keeps co

In [ ]:
ds_sampled

Dataset({
    features: ['input'],
    num_rows: 300
})

In [ ]:
ds_sampled.push_to_hub("imhann/medical_prompts", private=True)

# 2. Legal prompts dataset

In [ ]:
dataset = load_and_clean("jonathanli/legal-advice-reddit", "body")

In [ ]:
dataset

Dataset({
    features: ['body'],
    num_rows: 98910
})

In [ ]:
ds_filtered = filter_advice_dataset(dataset, text_column="body")

In [ ]:
ds_filtered

Dataset({
    features: ['body'],
    num_rows: 25520
})

In [ ]:
# sample 300 examples
ds_sampled = ds_filtered.shuffle(42).select(range(300))
ds_sampled[:5]

{'body': ['Long story short I’m (14) and have seen lolicon only for a few seconds on accident, im very stressed and have been googling stuff like “can anime be illegal” ever since it happened a few weeks ago. I need help from anyone that can possibly clear my mind up, Im in deep fear and hope someone can help me. I didn’t screenshot anything, send pictures or anything. All that happened was I searched the name of the show (which I thought was a normal anime) and clicked on the link which I now know was meant for hentai. I’ve seen porn before, and even hentai, but nothing like this and it wasn’t intentional. Again if anyone can help me out I would appreciate it dearly, please and thank you, if there’s any questions please let me know\nLive in U.S. btw',
  '(North Carolina) I was laid off due to Covid-19, my boss asked me to return 6 weeks later, including "If you feel more comfortable staying home now, that\'s okay too", The next day I told him I didn\'t want to be fired, but felt anoth

In [ ]:
ds_sampled

Dataset({
    features: ['body'],
    num_rows: 300
})

In [ ]:
ds_sampled.push_to_hub("imhann/legal_prompts", private=True)

# 3. Financial prompts dataset

In [ ]:
dataset = load_and_clean("llamafactory/fiqa", "input")

In [ ]:
dataset

Dataset({
    features: ['input'],
    num_rows: 6500
})

In [ ]:
ds_filtered = filter_advice_dataset(dataset, text_column="input")

In [ ]:
ds_filtered

Dataset({
    features: ['input'],
    num_rows: 616
})

In [ ]:
# sample 300 examples
ds_sampled = ds_filtered.shuffle(42).select(range(300))
ds_sampled[:5]

{'input': ['How can it be possible that only ~10% of options expire worthless, and only ~10% are exercised?',
  'Should I close unused credit cards before applying for another?',
  'How can I save LLC fees when investing in Arizona real estate from California',
  'When should I start saving/investing for my retirement?',
  "What should I look for when looking for stocks that are 'on-sale'?"]}

In [ ]:
ds_sampled

Dataset({
    features: ['input'],
    num_rows: 300
})

In [ ]:
ds_sampled.push_to_hub("imhann/financial_prompts", private=True)

# 4. Therapy/Counseling prompts dataset

In [ ]:
dataset = load_dataset("Psychotherapy-LLM/PsychoCounsel-Preference")

# merge possible train/test splits
dataset = concatenate_datasets([dataset[split] for split in dataset.keys()])

# keep only unique prompts using "ID" column, since prompts are repeated in the dataset
seen = set()

def is_unique(example):
    val = example["ID"]
    if val in seen:
        return False
    seen.add(val)
    return True

dataset = dataset.filter(is_unique)

# keep only prompt column
dataset = dataset.remove_columns(
  [col for col in dataset.column_names if col != "question"]
)

In [ ]:
dataset

Dataset({
    features: ['question'],
    num_rows: 17400
})

In [ ]:
ds_filtered = filter_advice_dataset(dataset, text_column="question")

In [ ]:
ds_filtered

Dataset({
    features: ['question'],
    num_rows: 2404
})

In [ ]:
# sample 300 examples
ds_sampled = ds_filtered.shuffle(42).select(range(300))
ds_sampled[:5]

{'question': ["I talked to some people at school and they suggested that I should see someone outside because there's a lot going on in my life. I recently went out, met someone, and it didn't go well. I met a guy who wasn't a good person, and it has made things hard for me.",
  "I have been with a guy for 4 years,  he's a great guy and we also have a son together. The problem is that I'm in love with a guy that I've been talking to for about 2 years but I've never met him in person. Honestly I'm bored with the relationship I have with the first guy and he makes want to go after the second guy, I don't know how to tell him that. What should I do?",
  "My recent breakup has left me feeling lost and heartbroken. The pain is overwhelming, and I can't seem to move on. I hope you can provide support and guidance so that I can heal, rediscover myself, and eventually open up to new relationships.",
  "Parenting has been incredibly difficult for me, especially as my child enters their teenage 

In [ ]:
ds_sampled

Dataset({
    features: ['question'],
    num_rows: 300
})

In [ ]:
ds_sampled.push_to_hub("imhann/therapy_prompts", private=True)

# Final concatenated dataset

In [ ]:
# concatenate the filtered and sampled datasets
ds1 = load_dataset("imhann/medical_prompts")
ds2 = load_dataset("imhann/legal_prompts")
ds3 = load_dataset("imhann/financial_prompts")
ds4 = load_dataset("imhann/therapy_prompts")

# rename the 'body' and 'question' as 'input'
ds2['train'] = ds2['train'].rename_column('body', 'input')
ds4['train'] = ds4['train'].rename_column('question', 'input')

# add column for prompt type
ds1['train'] = ds1['train'].add_column('prompt_type', ['medical'] * len(ds1['train']))
ds2['train'] = ds2['train'].add_column('prompt_type', ['legal'] * len(ds2['train']))
ds3['train'] = ds3['train'].add_column('prompt_type', ['financial'] * len(ds3['train']))
ds4['train'] = ds4['train'].add_column('prompt_type', ['therapy'] * len(ds4['train']))

# concatenate the datasets
ds_final = concatenate_datasets([ds1['train'], ds2['train'], ds3['train'], ds4['train']])


In [ ]:
ds_final

Dataset({
    features: ['input', 'prompt_type'],
    num_rows: 1200
})

In [ ]:
# save to hf
ds_final.push_to_hub("imhann/prompts_dataset", private=True)

In [ ]:
# get a random example from each 'promp_type'
ds = load_dataset("imhann/prompts_dataset", split='train')

types = set(ds['prompt_type'])

for t in types:
  subset = ds.filter(lambda x: x["prompt_type"] == t)
  example = subset.shuffle(seed=random.randint(0, 1_000_000)).select([0])
  print(example[0]['input'])

Filter:   0%|          | 0/1200 [00:00<?, ? examples/s]

Should I pay off a 0% car loan?


Filter:   0%|          | 0/1200 [00:00<?, ? examples/s]

On October 29, my employer had me arrested for Petit Larceny. Due to COVID my initial appearance was pushed back 3 times and I finally saw a judge today and entered a not guilty plea.

The state wants me to plea for 5days of jail time (I was held for 3 before I got out when arrested due to FTA traffic warrant that has since been rectified) and $641. However, my employer withheld my last check which was $675 and never gave me a pay stub for it. I was on the clock that day and was arrested at work. I was coerced by my immediate supervisor to manipulate tips in order to cover a shortage and am being charged with stealing tips, I was also told by my immediate supervisor that the $500 that was missing was covered and I can pay him back. Show up to work and arrested.

Long story short, I am a 46 year old mother of three and am looking to avoid this jail time. I have a public defender because I can’t afford a lawyer. My former employer knew I had no car yet had me working at a location where 

Filter:   0%|          | 0/1200 [00:00<?, ? examples/s]

I’m in a relationship from last 2 months. His behavior with me is mostly sweet/humble and kind. He is very helping, caring and charming. So, the problem is he gets angry very easily on lil matters.He tells me to be more sensible to say what should be said to do what should be done(according to him).I don’t have habit of thinking so much.We discuss for hours on my faults, he makes me realise.I apologise I promise to not repeat But then how will I know what can make him angry? His getting angry makes me cry coz I never want to hurt my sweetheart but it just happens. Ex- he wanted 2 know about my past experiences and I hesitate a bit being open quickly so he got hurt,he felt I wanna lie and all, but I never thought 2 lie. In the end he had 2 support me and help me overcome hesitation. But he was just very angry before understanding he needs to support.


Filter:   0%|          | 0/1200 [00:00<?, ? examples/s]

I have bad abdominal pain since after  breakfast.  I had a piece of toast with peanut butter and half a banana.  The pain is located throughout the whole stomach area and felt through the back.  It is a cramping pain and moving is difficult.  Should I go to emergency?  Thanks for your advice.
